Redes Bayesianas, Directo por Frecuencias y Naive Bayes
Objetivo: Codificar los ejercicios usando el método directo por frecuencias y el modelo Naive Bayes.
Además, implementar una red bayesiana (estructura tipo Naive) y mostrar las tablas de distribución (CPD/CPT).

In [2]:
import pandas as pd
import numpy as np

def fit_by_frequencies(df, target, features, laplace=1):
    """
    Estima:
      P(target) y P(feature=value | target)
    usando conteo directo por frecuencias + suavizado de Laplace.
    """
    target_counts = df[target].value_counts()
    classes = target_counts.index.tolist()

    # Prior P(target)
    prior = (target_counts + laplace) / (target_counts.sum() + laplace * len(classes))

    # Condicionales P(feature | target)
    cond = {}
    for f in features:
        ct = pd.crosstab(df[f], df[target])   # conteos
        ct = ct.reindex(columns=classes, fill_value=0)

        ct_lap = ct + laplace                # Laplace
        col_sums = ct_lap.sum(axis=0)
        cond[f] = ct_lap.div(col_sums, axis=1)

    return prior, cond


def predict_naive_bayes(prior, cond, evidence):
    """
    Calcula posterior proporcional a:
      P(class)* Π P(feature=value | class)
    """
    classes = prior.index.tolist()
    logp = {c: np.log(prior[c]) for c in classes}

    for f, v in evidence.items():
        if v not in cond[f].index:
            raise ValueError(f"Valor '{v}' no existe en '{f}'. Valores: {cond[f].index.tolist()}")
        for c in classes:
            logp[c] += np.log(cond[f].loc[v, c])

    # normalizar
    maxlog = max(logp.values())
    probs = {c: np.exp(logp[c] - maxlog) for c in classes}
    s = sum(probs.values())
    probs = {c: probs[c]/s for c in classes}

    return pd.Series(probs).sort_values(ascending=False)


def print_tables(prior, cond, target):
    print("=== Tabla de distribución P({}) ===".format(target))
    print(prior)
    for f, tbl in cond.items():
        print("\n=== Tabla de distribución P({} | {}) ===".format(f, target))
        print(tbl)


# Ejercicio 1: Compra (C)


In [5]:
df1 = pd.DataFrame([
    # C,   P,      G,           CP,    Pl,       A
    ["Sí","Bajo",  "Acción",     "Alta","PC",      "Sí"],
    ["Sí","Medio", "Aventura",   "Alta","Consola", "Sí"],
    ["No","Alto",  "Estrategia", "Baja","PC",      "No"],
    ["Sí","Bajo",  "Otros",      "Alta","PC",      "Sí"],
    ["No","Alto",  "Acción",     "Baja","Consola", "No"],
    ["Sí","Medio", "Acción",     "Alta","PC",      "Sí"],
    ["No","Medio", "Estrategia", "Baja","PC",      "No"],
    ["Sí","Bajo",  "Aventura",   "Alta","Consola", "Sí"],
], columns=["C","P","G","CP","Pl","A"])

target1 = "C"
features1 = ["P","G","CP","Pl","A"]

prior1, cond1 = fit_by_frequencies(df1, target1, features1, laplace=1)

print("DATASET (Ejercicio 1):")
display(df1)

print_tables(prior1, cond1, target1)

DATASET (Ejercicio 1):


,C,P,G,CP,Pl,A
0,Sí,Bajo,Acción,Alta,PC,Sí
1,Sí,Medio,Aventura,Alta,Consola,Sí
2,No,Alto,Estrategia,Baja,PC,No
3,Sí,Bajo,Otros,Alta,PC,Sí
4,No,Alto,Acción,Baja,Consola,No
5,Sí,Medio,Acción,Alta,PC,Sí
6,No,Medio,Estrategia,Baja,PC,No
7,Sí,Bajo,Aventura,Alta,Consola,Sí


=== Tabla de distribución P(C) ===
C
Sí    0.6
No    0.4
Name: count, dtype: float64

=== Tabla de distribución P(P | C) ===
C         Sí        No
P                     
Alto   0.125  0.500000
Bajo   0.500  0.166667
Medio  0.375  0.333333

=== Tabla de distribución P(G | C) ===
C                 Sí        No
G                             
Acción      0.333333  0.285714
Aventura    0.333333  0.142857
Estrategia  0.111111  0.428571
Otros       0.222222  0.142857

=== Tabla de distribución P(CP | C) ===
C           Sí   No
CP                 
Alta  0.857143  0.2
Baja  0.142857  0.8

=== Tabla de distribución P(Pl | C) ===
C              Sí   No
Pl                    
Consola  0.428571  0.4
PC       0.571429  0.6

=== Tabla de distribución P(A | C) ===
C         Sí   No
A                
No  0.142857  0.8
Sí  0.857143  0.2


In [7]:
evidence1 = {"P":"Medio", "G":"Acción", "CP":"Alta", "Pl":"PC", "A":"Sí"}

post1 = predict_naive_bayes(prior1, cond1, evidence1)

print("Evidencia:", evidence1)
print("\nPosterior P(C | evidencia):")
print(post1)
print("\nPredicción:", post1.index[0])

Evidencia: {'P': 'Medio', 'G': 'Acción', 'CP': 'Alta', 'Pl': 'PC', 'A': 'Sí'}

Posterior P(C | evidencia):
Sí    0.971782
No    0.028218
dtype: float64

Predicción: Sí


# Ejercicio 2: Éxito de la misión (EM)

In [10]:
df2 = pd.DataFrame([
    # EM,  NE,     EQ,           H,       S,      EMe
    ["Sí","Alto",  "Avanzado",    "Altas",  "Alta", "Positivo"],
    ["Sí","Medio", "Intermedio",  "Medias", "Media","Positivo"],
    ["No","Bajo",  "Básico",      "Bajas",  "Baja", "Negativo"],
    ["Sí","Alto",  "Intermedio",  "Altas",  "Alta", "Neutral"],
    ["No","Medio", "Básico",      "Medias", "Baja", "Negativo"],
    ["Sí","Medio", "Avanzado",    "Altas",  "Media","Positivo"],
    ["No","Bajo",  "Intermedio",  "Bajas",  "Media","Neutral"],
    ["Sí","Alto",  "Avanzado",    "Altas",  "Alta", "Positivo"],
], columns=["EM","NE","EQ","H","S","EMe"])

target2 = "EM"
features2 = ["NE","EQ","H","S","EMe"]

prior2, cond2 = fit_by_frequencies(df2, target2, features2, laplace=1)

print("DATASET (Ejercicio 2):")
display(df2)

print_tables(prior2, cond2, target2)


DATASET (Ejercicio 2):


,EM,NE,EQ,H,S,EMe
0,Sí,Alto,Avanzado,Altas,Alta,Positivo
1,Sí,Medio,Intermedio,Medias,Media,Positivo
2,No,Bajo,Básico,Bajas,Baja,Negativo
3,Sí,Alto,Intermedio,Altas,Alta,Neutral
4,No,Medio,Básico,Medias,Baja,Negativo
5,Sí,Medio,Avanzado,Altas,Media,Positivo
6,No,Bajo,Intermedio,Bajas,Media,Neutral
7,Sí,Alto,Avanzado,Altas,Alta,Positivo


=== Tabla de distribución P(EM) ===
EM
Sí    0.6
No    0.4
Name: count, dtype: float64

=== Tabla de distribución P(NE | EM) ===
EM        Sí        No
NE                    
Alto   0.500  0.166667
Bajo   0.125  0.500000
Medio  0.375  0.333333

=== Tabla de distribución P(EQ | EM) ===
EM             Sí        No
EQ                         
Avanzado    0.500  0.166667
Básico      0.125  0.500000
Intermedio  0.375  0.333333

=== Tabla de distribución P(H | EM) ===
EM         Sí        No
H                      
Altas   0.625  0.166667
Bajas   0.125  0.500000
Medias  0.250  0.333333

=== Tabla de distribución P(S | EM) ===
EM        Sí        No
S                     
Alta   0.500  0.166667
Baja   0.125  0.500000
Media  0.375  0.333333

=== Tabla de distribución P(EMe | EM) ===
EM           Sí        No
EMe                      
Negativo  0.125  0.500000
Neutral   0.250  0.333333
Positivo  0.625  0.166667


In [12]:
evidence2 = {"NE":"Medio", "EQ":"Intermedio", "H":"Medias", "S":"Media", "EMe":"Neutral"}

post2 = predict_naive_bayes(prior2, cond2, evidence2)

print("Evidencia:", evidence2)
print("\nPosterior P(EM | evidencia):")
print(post2)
print("\nPredicción:", post2.index[0])


Evidencia: {'NE': 'Medio', 'EQ': 'Intermedio', 'H': 'Medias', 'S': 'Media', 'EMe': 'Neutral'}

Posterior P(EM | evidencia):
Sí    0.545734
No    0.454266
dtype: float64

Predicción: Sí


# Implementación de la Red Bayesiana (pgmpy)
Se define el grafo y se cargan CPDs (tablas) calculadas por frecuencias.

In [19]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

def build_naive_bn(target, features):
    # Estructura: target -> cada feature
    edges = [(target, f) for f in features]
    return DiscreteBayesianNetwork(edges)

def cpds_from_tables(prior, cond, target):
    """
    Convierte:
      prior: Series P(target)
      cond: dict {feature: DataFrame index=estados feature, cols=clases} con P(feature|target)
    a CPDs de pgmpy.
    """
    classes = prior.index.tolist()
    cpds = []

    # CPD de la clase P(target)
    cpd_target = TabularCPD(
        variable=target,
        variable_card=len(classes),
        values=[[prior[c]] for c in classes],
        state_names={target: classes}
    )
    cpds.append(cpd_target)

    # CPD de cada feature P(feature | target)
    for f, tbl in cond.items():
        f_states = tbl.index.tolist()

        # pgmpy espera: filas = estados del feature, columnas = estados del padre (target)
        values = [tbl.loc[s, classes].tolist() for s in f_states]

        cpd_f = TabularCPD(
            variable=f,
            variable_card=len(f_states),
            values=values,
            evidence=[target],
            evidence_card=[len(classes)],
            state_names={f: f_states, target: classes}
        )
        cpds.append(cpd_f)

    return cpds


# -------------------------
# BN Ejercicio 1
# -------------------------
bn1 = build_naive_bn(target1, features1)
bn1.add_cpds(*cpds_from_tables(prior1, cond1, target1))

print("BN1 válido:", bn1.check_model())

infer1 = VariableElimination(bn1)
q1 = infer1.query(variables=[target1], evidence=evidence1, show_progress=False)
print("\nInferencia BN Ej1: P(C | evidencia)")
print(q1)


# -------------------------
# BN Ejercicio 2
# -------------------------
bn2 = build_naive_bn(target2, features2)
bn2.add_cpds(*cpds_from_tables(prior2, cond2, target2))

print("\nBN2 válido:", bn2.check_model())

infer2 = VariableElimination(bn2)
q2 = infer2.query(variables=[target2], evidence=evidence2, show_progress=False)
print("\nInferencia BN Ej2: P(EM | evidencia)")
print(q2)

BN1 válido: True

Inferencia BN Ej1: P(C | evidencia)
+-------+----------+
| C     |   phi(C) |
+=======+==========+
| C(Sí) |   0.9718 |
+-------+----------+
| C(No) |   0.0282 |
+-------+----------+

BN2 válido: True

Inferencia BN Ej2: P(EM | evidencia)
+--------+-----------+
| EM     |   phi(EM) |
+========+===========+
| EM(Sí) |    0.5457 |
+--------+-----------+
| EM(No) |    0.4543 |
+--------+-----------+
